In [1]:
import pandas as pd 
import requests
from jsonschema import validate
import json
import fastparquet

In [2]:
pd.set_option('display.width', None)

In [3]:
countriesDf = pd.DataFrame(pd.read_parquet('../dataRawBeforeBlob/countries/euCountriesRaw.parquet'))
countriesDf

,Code,Title,ParentDimension,Dimension,ParentCode,ParentTitle
0,AUT,Austria,REGION,COUNTRY,EUR,Europe
1,BEL,Belgium,REGION,COUNTRY,EUR,Europe
2,BGR,Bulgaria,REGION,COUNTRY,EUR,Europe
3,CYP,Cyprus,REGION,COUNTRY,EUR,Europe
4,CZE,Czechia,REGION,COUNTRY,EUR,Europe
5,DEU,Germany,REGION,COUNTRY,EUR,Europe
6,DNK,Denmark,REGION,COUNTRY,EUR,Europe
7,ESP,Spain,REGION,COUNTRY,EUR,Europe
8,EST,Estonia,REGION,COUNTRY,EUR,Europe
9,FIN,Finland,REGION,COUNTRY,EUR,Europe


In [11]:
baseURL= 'https://ghoapi.azureedge.net/api/'

In [7]:
ncdIndicatorsDf = pd.DataFrame(pd.read_parquet('../dataRawBeforeBlob/indicators/ncdIndicators.parquet',engine='fastparquet'))
ncdIndicatorsDf

,IndicatorCode,IndicatorName,Language,Category
0,NCD_CHOL_MEANTOTALCHOL_A,"Mean total cholesterol, age-standardized",EN,NCD: Risk factors and conditions
1,NCD_HYP_PREVALENCE_A,"Hypertension among adults aged 30-79 years, pr...",EN,NCD: Risk factors and conditions
2,NCD_DIABETES_PREVALENCE_AGESTD,"Prevalence of diabetes, age-standardized",EN,NCD: Risk factors and conditions
3,WHS2_131,Age-standardized NCD mortality rate (per 100 ...,EN,NCD: Mortality
4,NCD_DTH_TOT,Total NCD Deaths,EN,NCD: Mortality
5,NCDMORT3070,Probability (%) of dying between age 30 and ex...,EN,NCD: Mortality


In [8]:
def getData(baseurl,indicatorCode:str):
    indicatorCode = indicatorCode.strip()
    response = requests.get(baseurl+f"{indicatorCode}")
    if response.status_code == 200 and response.headers.get('Content-Type').startswith('application/json') :
        return response.json()
    else:
        return 'Something went wrong with request'

In [36]:
def saveToParquet(df: pd.DataFrame,filename:str):
    df.to_parquet(f'../dataRawBeforeBlob/ncdData/{filename}.parquet',engine='fastparquet')

In [10]:
firstIndicator = ncdIndicatorsDf.iloc[0]
firstIndicator

IndicatorCode                     NCD_CHOL_MEANTOTALCHOL_A
IndicatorName    Mean total cholesterol,  age-standardized
Language                                                EN
Category                  NCD: Risk factors and conditions
Name: 0, dtype: object

In [23]:
firstIndicatorData = getData(baseURL,firstIndicator['IndicatorCode'])
firstIndicatorData = firstIndicatorData['value']

firstIndicatorData

[{'Id': 9455120,
  'IndicatorCode': 'NCD_CHOL_MEANTOTALCHOL_A',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'TTO',
  'ParentLocationCode': 'AMR',
  'TimeDimType': 'YEAR',
  'ParentLocation': 'Americas',
  'Dim1Type': 'SEX',
  'TimeDim': 2006,
  'Dim1': 'SEX_MLE',
  'Dim2Type': None,
  'Dim2': None,
  'Dim3Type': None,
  'Dim3': None,
  'DataSourceDimType': None,
  'DataSourceDim': None,
  'Value': '4.5 [4.0-5.0]',
  'NumericValue': 4.5,
  'Low': 4.0,
  'High': 5.0,
  'Comments': None,
  'Date': '2021-06-21T18:29:13+02:00',
  'TimeDimensionValue': '2006',
  'TimeDimensionBegin': '2006-01-01T00:00:00+01:00',
  'TimeDimensionEnd': '2006-12-31T00:00:00+01:00'},
 {'Id': 9455336,
  'IndicatorCode': 'NCD_CHOL_MEANTOTALCHOL_A',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'MRT',
  'ParentLocationCode': 'AFR',
  'TimeDimType': 'YEAR',
  'ParentLocation': 'Africa',
  'Dim1Type': 'SEX',
  'TimeDim': 1990,
  'Dim1': 'SEX_MLE',
  'Dim2Type': None,
  'Dim2': None,
  'Dim3Type': None,
  'Dim3': 

In [16]:
firstIndicatorDataDataFrame = pd.DataFrame(firstIndicatorData)
firstIndicatorDataDataFrame = firstIndicatorDataDataFrame.loc[firstIndicatorDataDataFrame['SpatialDim'].isin(countriesDf['Code'])]
firstIndicatorDataDataFrame = firstIndicatorDataDataFrame.loc[firstIndicatorDataDataFrame['Dim1'] == 'SEX_BTSX']
firstIndicatorDataDataFrame['Value'] = firstIndicatorDataDataFrame['Value'].astype('string')
firstIndicatorDataDataFrame['Value'] = firstIndicatorDataDataFrame['Value'].str.split(' ').str[0]
firstIndicatorDataDataFrame['Value'] = firstIndicatorDataDataFrame['Value'].astype('float')
firstIndicatorDataDataFrame = firstIndicatorDataDataFrame[['Id','IndicatorCode','SpatialDim','TimeDim','Dim1','Value']]
firstIndicatorDataDataFrame

,Id,IndicatorCode,SpatialDim,TimeDim,Dim1,Value
46,20433,NCD_CHOL_MEANTOTALCHOL_A,AUT,1997,SEX_BTSX,5.6
108,48406,NCD_CHOL_MEANTOTALCHOL_A,LTU,2004,SEX_BTSX,5.5
188,82381,NCD_CHOL_MEANTOTALCHOL_A,ITA,2006,SEX_BTSX,5.2
217,96321,NCD_CHOL_MEANTOTALCHOL_A,DEU,2018,SEX_BTSX,4.9
218,98062,NCD_CHOL_MEANTOTALCHOL_A,POL,2018,SEX_BTSX,4.9
...,...,...,...,...,...,...
23534,10168983,NCD_CHOL_MEANTOTALCHOL_A,NLD,2008,SEX_BTSX,5.0
23536,10169626,NCD_CHOL_MEANTOTALCHOL_A,FRA,2009,SEX_BTSX,5.2
23546,10174045,NCD_CHOL_MEANTOTALCHOL_A,HRV,1990,SEX_BTSX,5.5
23559,10177994,NCD_CHOL_MEANTOTALCHOL_A,CYP,1995,SEX_BTSX,5.4


In [39]:
saveToParquet(firstIndicatorDataDataFrame,'ncdCholesterol')

In [18]:
secondIndicator = ncdIndicatorsDf.iloc[1]
secondIndicator

IndicatorCode                                 NCD_HYP_PREVALENCE_A
IndicatorName    Hypertension among adults aged 30-79 years, pr...
Language                                                        EN
Category                          NCD: Risk factors and conditions
Name: 1, dtype: object

In [19]:
secondIndicatorData = getData(baseURL,secondIndicator['IndicatorCode'])
secondIndicatorData = secondIndicatorData['value']
secondIndicatorData

[{'Id': 9455017,
  'IndicatorCode': 'NCD_HYP_PREVALENCE_A',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'VEN',
  'ParentLocationCode': 'AMR',
  'TimeDimType': 'YEAR',
  'ParentLocation': 'Americas',
  'Dim1Type': 'SEX',
  'TimeDim': 2014,
  'Dim1': 'SEX_MLE',
  'Dim2Type': None,
  'Dim2': None,
  'Dim3Type': None,
  'Dim3': None,
  'DataSourceDimType': None,
  'DataSourceDim': None,
  'Value': '40.4 [33.7-47.5]',
  'NumericValue': 40.4,
  'Low': 33.7,
  'High': 47.5,
  'Comments': None,
  'Date': '2021-08-20T16:07:55+02:00',
  'TimeDimensionValue': '2014',
  'TimeDimensionBegin': '2014-01-01T00:00:00+01:00',
  'TimeDimensionEnd': '2014-12-31T00:00:00+01:00'},
 {'Id': 1558,
  'IndicatorCode': 'NCD_HYP_PREVALENCE_A',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'UKR',
  'ParentLocationCode': 'EUR',
  'TimeDimType': 'YEAR',
  'ParentLocation': 'Europe',
  'Dim1Type': 'SEX',
  'TimeDim': 2003,
  'Dim1': 'SEX_BTSX',
  'Dim2Type': None,
  'Dim2': None,
  'Dim3Type': None,
  'Dim3': None

In [20]:
secondIndicatorDataDataFrame = pd.DataFrame(secondIndicatorData)
secondIndicatorDataDataFrame = secondIndicatorDataDataFrame.loc[secondIndicatorDataDataFrame['SpatialDim'].isin(countriesDf['Code'])]
secondIndicatorDataDataFrame = secondIndicatorDataDataFrame.loc[secondIndicatorDataDataFrame['Dim1'] == 'SEX_BTSX']
secondIndicatorDataDataFrame['Value'] = secondIndicatorDataDataFrame['Value'].astype('string')
secondIndicatorDataDataFrame['Value'] = secondIndicatorDataDataFrame['Value'].str.split(' ').str[0]
secondIndicatorDataDataFrame['Value'] = secondIndicatorDataDataFrame['Value'].astype('float')
secondIndicatorDataDataFrame = secondIndicatorDataDataFrame[['Id','IndicatorCode','SpatialDim','TimeDim','Dim1','Value']]
secondIndicatorDataDataFrame

,Id,IndicatorCode,SpatialDim,TimeDim,Dim1,Value
34,14847,NCD_HYP_PREVALENCE_A,IRL,2002,SEX_BTSX,36.4
50,23641,NCD_HYP_PREVALENCE_A,SWE,2019,SEX_BTSX,30.2
64,28906,NCD_HYP_PREVALENCE_A,CYP,2009,SEX_BTSX,34.3
120,55148,NCD_HYP_PREVALENCE_A,SVN,1991,SEX_BTSX,47.4
139,66433,NCD_HYP_PREVALENCE_A,POL,2012,SEX_BTSX,47.3
...,...,...,...,...,...,...
19206,10192284,NCD_HYP_PREVALENCE_A,EST,2011,SEX_BTSX,41.8
19208,10194150,NCD_HYP_PREVALENCE_A,LUX,2016,SEX_BTSX,32.8
19212,10196554,NCD_HYP_PREVALENCE_A,FIN,2006,SEX_BTSX,40.2
19220,10201077,NCD_HYP_PREVALENCE_A,ESP,2005,SEX_BTSX,35.3


In [40]:
saveToParquet(secondIndicatorDataDataFrame,'ncdHypertension')

In [21]:
thirdIndicator = ncdIndicatorsDf.iloc[2]
thirdIndicator

IndicatorCode              NCD_DIABETES_PREVALENCE_AGESTD
IndicatorName    Prevalence of diabetes, age-standardized
Language                                               EN
Category                 NCD: Risk factors and conditions
Name: 2, dtype: object

In [22]:
thirdIndicatorData = getData(baseURL,thirdIndicator['IndicatorCode'])
thirdIndicatorData = thirdIndicatorData['value']
thirdIndicatorData

[{'Id': 9455041,
  'IndicatorCode': 'NCD_DIABETES_PREVALENCE_AGESTD',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'AFG',
  'ParentLocationCode': 'EMR',
  'TimeDimType': 'YEAR',
  'ParentLocation': 'Eastern Mediterranean',
  'Dim1Type': 'SEX',
  'Dim1': 'SEX_MLE',
  'TimeDim': 2003,
  'Dim2Type': 'AGEGROUP',
  'Dim2': 'AGEGROUP_YEARS18-PLUS',
  'Dim3Type': None,
  'Dim3': None,
  'DataSourceDimType': None,
  'DataSourceDim': None,
  'Value': '13.6 [6.5-23.1]',
  'NumericValue': 13.594753,
  'Low': 6.4873122,
  'High': 23.0599,
  'Comments': None,
  'Date': '2024-11-13T10:26:48.993+01:00',
  'TimeDimensionValue': '2003',
  'TimeDimensionBegin': '2003-01-01T00:00:00+01:00',
  'TimeDimensionEnd': '2003-12-31T00:00:00+01:00'},
 {'Id': 9455103,
  'IndicatorCode': 'NCD_DIABETES_PREVALENCE_AGESTD',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'SOM',
  'ParentLocationCode': 'EMR',
  'TimeDimType': 'YEAR',
  'ParentLocation': 'Eastern Mediterranean',
  'Dim1Type': 'SEX',
  'Dim1': 'SEX_MLE'

In [25]:
thirdIndicatorDataDataFrame = pd.DataFrame(thirdIndicatorData)
thirdIndicatorDataDataFrame = thirdIndicatorDataDataFrame.loc[thirdIndicatorDataDataFrame['SpatialDim'].isin(countriesDf['Code'])]
thirdIndicatorDataDataFrame = thirdIndicatorDataDataFrame[(thirdIndicatorDataDataFrame['Dim1'] == 'SEX_BTSX')]
thirdIndicatorDataDataFrame['Value'] = thirdIndicatorDataDataFrame['Value'].astype('string')
thirdIndicatorDataDataFrame['Value'] = thirdIndicatorDataDataFrame['Value'].str.split(' ').str[0]
thirdIndicatorDataDataFrame['Value'] = thirdIndicatorDataDataFrame['Value'].astype('float')
thirdIndicatorDataDataFrame = thirdIndicatorDataDataFrame[['Id','IndicatorCode','SpatialDim','TimeDim','Dim1','Dim2','Value']]
thirdIndicatorDataDataFrame

,Id,IndicatorCode,SpatialDim,TimeDim,Dim1,Dim2,Value
9,959,NCD_DIABETES_PREVALENCE_AGESTD,LTU,2002,SEX_BTSX,AGEGROUP_YEARS18-PLUS,7.2
23,5429,NCD_DIABETES_PREVALENCE_AGESTD,ROU,2002,SEX_BTSX,AGEGROUP_YEARS30-PLUS,5.1
28,6337,NCD_DIABETES_PREVALENCE_AGESTD,DNK,2001,SEX_BTSX,AGEGROUP_YEARS30-PLUS,4.0
36,8438,NCD_DIABETES_PREVALENCE_AGESTD,SVN,2002,SEX_BTSX,AGEGROUP_YEARS30-PLUS,8.3
56,14257,NCD_DIABETES_PREVALENCE_AGESTD,AUT,1991,SEX_BTSX,AGEGROUP_YEARS18-PLUS,3.7
...,...,...,...,...,...,...,...
41475,10181482,NCD_DIABETES_PREVALENCE_AGESTD,SVK,2019,SEX_BTSX,AGEGROUP_YEARS30-PLUS,10.7
41491,10185239,NCD_DIABETES_PREVALENCE_AGESTD,SVK,1995,SEX_BTSX,AGEGROUP_YEARS30-PLUS,7.2
41508,10189473,NCD_DIABETES_PREVALENCE_AGESTD,HUN,2006,SEX_BTSX,AGEGROUP_YEARS30-PLUS,10.1
41523,10193666,NCD_DIABETES_PREVALENCE_AGESTD,ITA,2022,SEX_BTSX,AGEGROUP_YEARS18-PLUS,7.2


In [27]:
thirdIndicatorDataDataFrame18Plus = thirdIndicatorDataDataFrame.loc[thirdIndicatorDataDataFrame['Dim2'] == 'AGEGROUP_YEARS18-PLUS']
thirdIndicatorDataDataFrame18Plus

,Id,IndicatorCode,SpatialDim,TimeDim,Dim1,Dim2,Value
9,959,NCD_DIABETES_PREVALENCE_AGESTD,LTU,2002,SEX_BTSX,AGEGROUP_YEARS18-PLUS,7.2
56,14257,NCD_DIABETES_PREVALENCE_AGESTD,AUT,1991,SEX_BTSX,AGEGROUP_YEARS18-PLUS,3.7
62,14911,NCD_DIABETES_PREVALENCE_AGESTD,SWE,2012,SEX_BTSX,AGEGROUP_YEARS18-PLUS,4.3
112,28962,NCD_DIABETES_PREVALENCE_AGESTD,DNK,2010,SEX_BTSX,AGEGROUP_YEARS18-PLUS,2.8
144,36743,NCD_DIABETES_PREVALENCE_AGESTD,FIN,2018,SEX_BTSX,AGEGROUP_YEARS18-PLUS,6.6
...,...,...,...,...,...,...,...
41160,10107730,NCD_DIABETES_PREVALENCE_AGESTD,DEU,2005,SEX_BTSX,AGEGROUP_YEARS18-PLUS,6.8
41226,10123830,NCD_DIABETES_PREVALENCE_AGESTD,GRC,1992,SEX_BTSX,AGEGROUP_YEARS18-PLUS,5.5
41405,10166506,NCD_DIABETES_PREVALENCE_AGESTD,SVK,1996,SEX_BTSX,AGEGROUP_YEARS18-PLUS,5.3
41414,10169143,NCD_DIABETES_PREVALENCE_AGESTD,BGR,2014,SEX_BTSX,AGEGROUP_YEARS18-PLUS,8.3


In [28]:
thirdIndicatorDataDataFrame30Plus = thirdIndicatorDataDataFrame.loc[thirdIndicatorDataDataFrame['Dim2'] == 'AGEGROUP_YEARS30-PLUS']
thirdIndicatorDataDataFrame30Plus

,Id,IndicatorCode,SpatialDim,TimeDim,Dim1,Dim2,Value
23,5429,NCD_DIABETES_PREVALENCE_AGESTD,ROU,2002,SEX_BTSX,AGEGROUP_YEARS30-PLUS,5.1
28,6337,NCD_DIABETES_PREVALENCE_AGESTD,DNK,2001,SEX_BTSX,AGEGROUP_YEARS30-PLUS,4.0
36,8438,NCD_DIABETES_PREVALENCE_AGESTD,SVN,2002,SEX_BTSX,AGEGROUP_YEARS30-PLUS,8.3
58,14317,NCD_DIABETES_PREVALENCE_AGESTD,DEU,1999,SEX_BTSX,AGEGROUP_YEARS30-PLUS,8.6
160,41599,NCD_DIABETES_PREVALENCE_AGESTD,EST,2003,SEX_BTSX,AGEGROUP_YEARS30-PLUS,6.6
...,...,...,...,...,...,...,...
41392,10164874,NCD_DIABETES_PREVALENCE_AGESTD,SWE,2017,SEX_BTSX,AGEGROUP_YEARS30-PLUS,6.0
41475,10181482,NCD_DIABETES_PREVALENCE_AGESTD,SVK,2019,SEX_BTSX,AGEGROUP_YEARS30-PLUS,10.7
41491,10185239,NCD_DIABETES_PREVALENCE_AGESTD,SVK,1995,SEX_BTSX,AGEGROUP_YEARS30-PLUS,7.2
41508,10189473,NCD_DIABETES_PREVALENCE_AGESTD,HUN,2006,SEX_BTSX,AGEGROUP_YEARS30-PLUS,10.1


In [34]:
mergedDf = pd.merge(thirdIndicatorDataDataFrame18Plus,thirdIndicatorDataDataFrame30Plus,on=['SpatialDim','TimeDim'])
mergedDf['meanValue'] = (mergedDf['Value_x'] + mergedDf['Value_y']) / 2
mergedDf['newDim2'] = 'AGEGROUP_ALLYEARS'
mergedDf = mergedDf[['Id_x','IndicatorCode_x','SpatialDim','TimeDim','Dim1_x','newDim2','meanValue']]
mergedDf


,Id_x,IndicatorCode_x,SpatialDim,TimeDim,Dim1_x,newDim2,meanValue
0,959,NCD_DIABETES_PREVALENCE_AGESTD,LTU,2002,SEX_BTSX,AGEGROUP_ALLYEARS,8.30
1,14257,NCD_DIABETES_PREVALENCE_AGESTD,AUT,1991,SEX_BTSX,AGEGROUP_ALLYEARS,4.35
2,14911,NCD_DIABETES_PREVALENCE_AGESTD,SWE,2012,SEX_BTSX,AGEGROUP_ALLYEARS,5.00
3,28962,NCD_DIABETES_PREVALENCE_AGESTD,DNK,2010,SEX_BTSX,AGEGROUP_ALLYEARS,3.25
4,36743,NCD_DIABETES_PREVALENCE_AGESTD,FIN,2018,SEX_BTSX,AGEGROUP_ALLYEARS,7.65
...,...,...,...,...,...,...,...
886,10107730,NCD_DIABETES_PREVALENCE_AGESTD,DEU,2005,SEX_BTSX,AGEGROUP_ALLYEARS,8.00
887,10123830,NCD_DIABETES_PREVALENCE_AGESTD,GRC,1992,SEX_BTSX,AGEGROUP_ALLYEARS,6.50
888,10166506,NCD_DIABETES_PREVALENCE_AGESTD,SVK,1996,SEX_BTSX,AGEGROUP_ALLYEARS,6.25
889,10169143,NCD_DIABETES_PREVALENCE_AGESTD,BGR,2014,SEX_BTSX,AGEGROUP_ALLYEARS,9.65


In [41]:
saveToParquet(mergedDf,'ncdDiabetes')

In [42]:
fourthIndicator = ncdIndicatorsDf.iloc[3]
fourthIndicator

IndicatorCode                                             WHS2_131
IndicatorName    Age-standardized NCD mortality rate  (per 100 ...
Language                                                        EN
Category                                            NCD: Mortality
Name: 3, dtype: object

In [44]:
fourthIndicatorData = getData(baseURL,fourthIndicator['IndicatorCode'])
fourthIndicatorData = fourthIndicatorData['value']
fourthIndicatorData

[{'Id': 2683,
  'IndicatorCode': 'WHS2_131',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'MNG',
  'ParentLocationCode': 'WPR',
  'TimeDimType': 'YEAR',
  'ParentLocation': 'Western Pacific',
  'Dim1Type': 'SEX',
  'TimeDim': 2008,
  'Dim1': 'SEX_MLE',
  'Dim2Type': None,
  'Dim2': None,
  'Dim3Type': None,
  'Dim3': None,
  'DataSourceDimType': None,
  'DataSourceDim': None,
  'Value': '1150 [800-1640]',
  'NumericValue': 1154.453613281,
  'Low': 795.569702148,
  'High': 1643.545288086,
  'Comments': None,
  'Date': '2025-03-19T11:00:50.48+01:00',
  'TimeDimensionValue': '2008',
  'TimeDimensionBegin': '2008-01-01T00:00:00+01:00',
  'TimeDimensionEnd': '2008-12-31T00:00:00+01:00'},
 {'Id': 7584,
  'IndicatorCode': 'WHS2_131',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'VNM',
  'ParentLocationCode': 'WPR',
  'TimeDimType': 'YEAR',
  'ParentLocation': 'Western Pacific',
  'Dim1Type': 'SEX',
  'TimeDim': 2002,
  'Dim1': 'SEX_BTSX',
  'Dim2Type': None,
  'Dim2': None,
  'Dim3Type': 

In [45]:
fourthIndicatorDataDataFrame = pd.DataFrame(fourthIndicatorData)
fourthIndicatorDataDataFrame = fourthIndicatorDataDataFrame.loc[fourthIndicatorDataDataFrame['SpatialDim'].isin(countriesDf['Code'])]
fourthIndicatorDataDataFrame = fourthIndicatorDataDataFrame.loc[fourthIndicatorDataDataFrame['Dim1'] == 'SEX_BTSX']
fourthIndicatorDataDataFrame['Value'] = fourthIndicatorDataDataFrame['Value'].astype('string')
fourthIndicatorDataDataFrame['Value'] = fourthIndicatorDataDataFrame['Value'].str.split(' ').str[0]
fourthIndicatorDataDataFrame['Value'] = fourthIndicatorDataDataFrame['Value'].astype('float')
fourthIndicatorDataDataFrame = fourthIndicatorDataDataFrame[['Id','IndicatorCode','SpatialDim','TimeDim','Dim1','Value']]
fourthIndicatorDataDataFrame

,Id,IndicatorCode,SpatialDim,TimeDim,Dim1,Value
18,71396,WHS2_131,CYP,2005,SEX_BTSX,400.0
48,178584,WHS2_131,SVK,2011,SEX_BTSX,550.0
65,242153,WHS2_131,ROU,2002,SEX_BTSX,770.0
96,356867,WHS2_131,BGR,2020,SEX_BTSX,650.0
106,382764,WHS2_131,CZE,2010,SEX_BTSX,480.0
...,...,...,...,...,...,...
12766,9636213,WHS2_131,CYP,2019,SEX_BTSX,310.0
12828,9812964,WHS2_131,PRT,2007,SEX_BTSX,400.0
12833,9831630,WHS2_131,DNK,2003,SEX_BTSX,490.0
12906,10122397,WHS2_131,HUN,2008,SEX_BTSX,650.0


In [48]:
saveToParquet(fourthIndicatorDataDataFrame,'ncdMortalityRatePer100000')

In [47]:
fifthIndicator = ncdIndicatorsDf.iloc[4]
fifthIndicator

IndicatorCode         NCD_DTH_TOT
IndicatorName    Total NCD Deaths
Language                       EN
Category           NCD: Mortality
Name: 4, dtype: object

In [49]:
fifthIndicatorData = getData(baseURL,fifthIndicator['IndicatorCode'])
fifthIndicatorData = fifthIndicatorData['value']
fifthIndicatorData

[{'Id': 9454999,
  'IndicatorCode': 'NCD_DTH_TOT',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'ISL',
  'ParentLocationCode': 'EUR',
  'TimeDimType': 'YEAR',
  'ParentLocation': 'Europe',
  'Dim1Type': 'SEX',
  'TimeDim': 2015,
  'Dim1': 'SEX_FMLE',
  'Dim2Type': 'GHECAUSES',
  'Dim2': 'GHECAUSES_GHE060',
  'Dim3Type': None,
  'Dim3': None,
  'DataSourceDimType': None,
  'DataSourceDim': None,
  'Value': '980 [700-1320]',
  'NumericValue': 975.0,
  'Low': 703.0,
  'High': 1319.0,
  'Comments': None,
  'Date': '2025-03-19T11:00:50.48+01:00',
  'TimeDimensionValue': '2015',
  'TimeDimensionBegin': '2015-01-01T00:00:00+01:00',
  'TimeDimensionEnd': '2015-12-31T00:00:00+01:00'},
 {'Id': 9455488,
  'IndicatorCode': 'NCD_DTH_TOT',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'POL',
  'ParentLocationCode': 'EUR',
  'TimeDimType': 'YEAR',
  'ParentLocation': 'Europe',
  'Dim1Type': 'SEX',
  'TimeDim': 2014,
  'Dim1': 'SEX_MLE',
  'Dim2Type': 'GHECAUSES',
  'Dim2': 'GHECAUSES_GHE060',
  'D

In [55]:
fifthIndicatorDataDataFrame = pd.DataFrame(fifthIndicatorData)
fifthIndicatorDataDataFrame = fifthIndicatorDataDataFrame.loc[fifthIndicatorDataDataFrame['SpatialDim'].isin(countriesDf['Code'])]
fifthIndicatorDataDataFrame = fifthIndicatorDataDataFrame.loc[fifthIndicatorDataDataFrame['Dim1'] == 'SEX_BTSX']
fifthIndicatorDataDataFrame['Value'] = fifthIndicatorDataDataFrame['Value'].astype('string')
fifthIndicatorDataDataFrame['Value'] = fifthIndicatorDataDataFrame['Value'].str.split(' ').str[0]
fifthIndicatorDataDataFrame['Value'] = fifthIndicatorDataDataFrame['Value'].astype('float')
fifthIndicatorDataDataFrame = fifthIndicatorDataDataFrame[['Id','IndicatorCode','SpatialDim','TimeDim','Dim1','Value']]
fifthIndicatorDataDataFrame

,Id,IndicatorCode,SpatialDim,TimeDim,Dim1,Value
37,27058,NCD_DTH_TOT,FRA,2001,SEX_BTSX,459.0
40,29167,NCD_DTH_TOT,HRV,2018,SEX_BTSX,47.0
42,29635,NCD_DTH_TOT,LVA,2008,SEX_BTSX,28.0
44,34390,NCD_DTH_TOT,CYP,2002,SEX_BTSX,5310.0
63,49028,NCD_DTH_TOT,SVK,2009,SEX_BTSX,47.0
...,...,...,...,...,...,...
12748,10066946,NCD_DTH_TOT,HRV,2010,SEX_BTSX,47.0
12827,10121434,NCD_DTH_TOT,SVN,2012,SEX_BTSX,16.0
12834,10125647,NCD_DTH_TOT,MLT,2001,SEX_BTSX,2700.0
12872,10154288,NCD_DTH_TOT,ROU,2015,SEX_BTSX,239.0


In [56]:
saveToParquet(fifthIndicatorDataDataFrame,'totalNCDdeaths')

In [57]:
sixthIndicator = ncdIndicatorsDf.iloc[5]
sixthIndicator

IndicatorCode                                          NCDMORT3070
IndicatorName    Probability (%) of dying between age 30 and ex...
Language                                                        EN
Category                                            NCD: Mortality
Name: 5, dtype: object

In [58]:
sixthIndicator['IndicatorName']

'Probability (%) of dying between age 30 and exact age 70 from any of cardiovascular disease, cancer, diabetes, or chronic respiratory disease'

In [59]:
sixthIndicatorData = getData(baseURL,sixthIndicator['IndicatorCode'])
sixthIndicatorData = sixthIndicatorData['value']
sixthIndicatorData

[{'Id': 4213,
  'IndicatorCode': 'NCDMORT3070',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'AGO',
  'TimeDimType': 'YEAR',
  'ParentLocationCode': 'AFR',
  'ParentLocation': 'Africa',
  'Dim1Type': 'SEX',
  'Dim1': 'SEX_FMLE',
  'TimeDim': 2000,
  'Dim2Type': 'AGEGROUP',
  'Dim2': 'AGEGROUP_YEARS30-69',
  'Dim3Type': None,
  'Dim3': None,
  'DataSourceDimType': None,
  'DataSourceDim': None,
  'Value': '27.8 [16.1-40.1]',
  'NumericValue': 27.8,
  'Low': 16.1,
  'High': 40.1,
  'Comments': None,
  'Date': '2024-12-18T14:54:42.07+01:00',
  'TimeDimensionValue': '2000',
  'TimeDimensionBegin': '2000-01-01T00:00:00+01:00',
  'TimeDimensionEnd': '2000-12-31T00:00:00+01:00'},
 {'Id': 8902,
  'IndicatorCode': 'NCDMORT3070',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'ZWE',
  'TimeDimType': 'YEAR',
  'ParentLocationCode': 'AFR',
  'ParentLocation': 'Africa',
  'Dim1Type': 'SEX',
  'Dim1': 'SEX_BTSX',
  'TimeDim': 2021,
  'Dim2Type': 'AGEGROUP',
  'Dim2': 'AGEGROUP_YEARS30-69',
  'Dim3

In [60]:
sixthIndicatorDataDataFrame = pd.DataFrame(sixthIndicatorData)
sixthIndicatorDataDataFrame

,Id,IndicatorCode,SpatialDimType,SpatialDim,TimeDimType,ParentLocationCode,ParentLocation,Dim1Type,Dim1,TimeDim,...,DataSourceDim,Value,NumericValue,Low,High,Comments,Date,TimeDimensionValue,TimeDimensionBegin,TimeDimensionEnd
0,4213,NCDMORT3070,COUNTRY,AGO,YEAR,AFR,Africa,SEX,SEX_FMLE,2000,...,None,27.8 [16.1-40.1],27.8,16.1,40.1,None,2024-12-18T14:54:42.07+01:00,2000,2000-01-01T00:00:00+01:00,2000-12-31T00:00:00+01:00
1,8902,NCDMORT3070,COUNTRY,ZWE,YEAR,AFR,Africa,SEX,SEX_BTSX,2021,...,None,31.2 [19.2-42.9],31.2,19.2,42.9,"For this geography and year, some non-communic...",2024-12-18T14:54:42.07+01:00,2021,2021-01-01T00:00:00+01:00,2021-12-31T00:00:00+01:00
2,8906,NCDMORT3070,COUNTRY,GMB,YEAR,AFR,Africa,SEX,SEX_MLE,2006,...,None,26.2 [16.5-37.7],26.2,16.5,37.7,None,2024-12-18T14:54:42.07+01:00,2006,2006-01-01T00:00:00+01:00,2006-12-31T00:00:00+01:00
3,8953,NCDMORT3070,COUNTRY,BRA,YEAR,AMR,Americas,SEX,SEX_BTSX,2020,...,None,14.7 [13.7-15.8],14.7,13.7,15.8,None,2024-12-18T14:54:42.07+01:00,2020,2020-01-01T00:00:00+01:00,2020-12-31T00:00:00+01:00
4,10610,NCDMORT3070,COUNTRY,AFG,YEAR,EMR,Eastern Mediterranean,SEX,SEX_BTSX,2003,...,None,42.5 [25.6-60.1],42.5,25.6,60.1,None,2024-12-18T14:54:42.07+01:00,2003,2003-01-01T00:00:00+01:00,2003-12-31T00:00:00+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12931,10198987,NCDMORT3070,COUNTRY,SSD,YEAR,AFR,Africa,SEX,SEX_MLE,2000,...,None,23.7 [14.3-37.0],23.7,14.3,37.0,None,2024-12-18T14:54:42.07+01:00,2000,2000-01-01T00:00:00+01:00,2000-12-31T00:00:00+01:00
12932,10203921,NCDMORT3070,COUNTRY,ISL,YEAR,EUR,Europe,SEX,SEX_BTSX,2015,...,None,9.2 [7.6-10.9],9.2,7.6,10.9,None,2024-12-18T14:54:42.07+01:00,2015,2015-01-01T00:00:00+01:00,2015-12-31T00:00:00+01:00
12933,10203922,NCDMORT3070,COUNTRY,HUN,YEAR,EUR,Europe,SEX,SEX_BTSX,2021,...,None,21.7 [18.3-25.2],21.7,18.3,25.2,None,2024-12-18T14:54:42.07+01:00,2021,2021-01-01T00:00:00+01:00,2021-12-31T00:00:00+01:00
12934,10204218,NCDMORT3070,COUNTRY,LBN,YEAR,EMR,Eastern Mediterranean,SEX,SEX_FMLE,2020,...,None,9.6 [6.4-13.6],9.6,6.4,13.6,"For this geography and year, some non-communic...",2024-12-18T14:54:42.07+01:00,2020,2020-01-01T00:00:00+01:00,2020-12-31T00:00:00+01:00


In [63]:
sixthIndicatorDataDataFrame = pd.DataFrame(sixthIndicatorData)
sixthIndicatorDataDataFrame = sixthIndicatorDataDataFrame.loc[sixthIndicatorDataDataFrame['SpatialDim'].isin(countriesDf['Code'])]
sixthIndicatorDataDataFrame = sixthIndicatorDataDataFrame.loc[sixthIndicatorDataDataFrame['Dim1'] == 'SEX_BTSX']
sixthIndicatorDataDataFrame['Value'] = sixthIndicatorDataDataFrame['Value'].astype('string')
sixthIndicatorDataDataFrame['Value'] = sixthIndicatorDataDataFrame['Value'].str.split(' ').str[0]
sixthIndicatorDataDataFrame['Value'] = sixthIndicatorDataDataFrame['Value'].astype('float')
sixthIndicatorDataDataFrame

,Id,IndicatorCode,SpatialDimType,SpatialDim,TimeDimType,ParentLocationCode,ParentLocation,Dim1Type,Dim1,TimeDim,...,DataSourceDim,Value,NumericValue,Low,High,Comments,Date,TimeDimensionValue,TimeDimensionBegin,TimeDimensionEnd
29,47715,NCDMORT3070,COUNTRY,AUT,YEAR,EUR,Europe,SEX,SEX_BTSX,2018,...,None,10.9,10.9,9.3,12.7,None,2024-12-18T14:54:42.07+01:00,2018,2018-01-01T00:00:00+01:00,2018-12-31T00:00:00+01:00
80,122203,NCDMORT3070,COUNTRY,LTU,YEAR,EUR,Europe,SEX,SEX_BTSX,2020,...,None,20.7,20.7,17.8,24.2,None,2024-12-18T14:54:42.07+01:00,2020,2020-01-01T00:00:00+01:00,2020-12-31T00:00:00+01:00
118,179831,NCDMORT3070,COUNTRY,CYP,YEAR,EUR,Europe,SEX,SEX_BTSX,2008,...,None,11.0,11.0,8.3,14.4,None,2024-12-18T14:54:42.07+01:00,2008,2008-01-01T00:00:00+01:00,2008-12-31T00:00:00+01:00
123,192234,NCDMORT3070,COUNTRY,HUN,YEAR,EUR,Europe,SEX,SEX_BTSX,2004,...,None,26.8,26.8,23.8,29.9,None,2024-12-18T14:54:42.07+01:00,2004,2004-01-01T00:00:00+01:00,2004-12-31T00:00:00+01:00
160,242748,NCDMORT3070,COUNTRY,PRT,YEAR,EUR,Europe,SEX,SEX_BTSX,2015,...,None,11.5,11.5,9.7,13.4,None,2024-12-18T14:54:42.07+01:00,2015,2015-01-01T00:00:00+01:00,2015-12-31T00:00:00+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12871,10108158,NCDMORT3070,COUNTRY,POL,YEAR,EUR,Europe,SEX,SEX_BTSX,2011,...,None,20.0,20.0,19.5,21.2,None,2024-12-18T14:54:42.07+01:00,2011,2011-01-01T00:00:00+01:00,2011-12-31T00:00:00+01:00
12873,10108861,NCDMORT3070,COUNTRY,PRT,YEAR,EUR,Europe,SEX,SEX_BTSX,2014,...,None,11.5,11.5,9.7,13.4,None,2024-12-18T14:54:42.07+01:00,2014,2014-01-01T00:00:00+01:00,2014-12-31T00:00:00+01:00
12876,10114556,NCDMORT3070,COUNTRY,DEU,YEAR,EUR,Europe,SEX,SEX_BTSX,2018,...,None,12.1,12.1,10.4,14.0,None,2024-12-18T14:54:42.07+01:00,2018,2018-01-01T00:00:00+01:00,2018-12-31T00:00:00+01:00
12922,10190211,NCDMORT3070,COUNTRY,LTU,YEAR,EUR,Europe,SEX,SEX_BTSX,2015,...,None,22.2,22.2,19.9,25.7,None,2024-12-18T14:54:42.07+01:00,2015,2015-01-01T00:00:00+01:00,2015-12-31T00:00:00+01:00


In [ ]:
saveToParquet(sixthIndicatorDataDataFrame,'probabilityOfDying30-70NCD')